# 🧪 Modelo Causal 1 baseado em TinyLlama

**Ajuste fino (fine-tuning)** do modelo de linguagem causal `TinyLlama/TinyLlama-1.1B-Chat-v1.0` utilizando **LoRA** (*Low-Rank Adaptation*), uma das técnicas de **PEFT** (*Parameter-Efficient Fine-Tuning*).  

## 1. Requisitos e Instalação

- Execute o comando para instalar as dependências necessárias:
- A linha abaixo referente ao torchao é utilizada para uso no COLAB

In [ ]:
!pip install transformers datasets peft accelerate torch sentencepiece

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    !pip install --upgrade torchao>=0.16.0

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
import torch

## 2. Carregar e Preparar o Dataset

Utilizaremos o arquivo `dataset.jsonl`, onde cada linha contém uma instrução (`Instruction`) e a saída desejada (`Output`).  
Vamos converter cada exemplo em uma única string no formato:
```
Instruction: <instrução>
Output: <saída>
```
e depois dividir o conjunto em treino (80%) e validação (20%).

In [ ]:
def convert_to_hf_format(example):
    """Combina instrução e saída em um único texto."""
    return {
        "text": f"Instruction: {example['Instruction']}\nOutput: {example['Output']}"
    }

# Carrega o arquivo JSON Lines (campos: "Instruction" e "Output")
dataset = load_dataset('json', data_files='dataset.jsonl')
# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)
# Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)

## 3. Carregar o Modelo Pré-Treinado e o Tokenizador

Vamos carregar o `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, um modelo de 1,1 bilhão de parâmetros baseado na arquitetura LLaMA.

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

## 4. Inferência ANTES do Fine-Tuning

Antes de qualquer treinamento, vamos ver como o modelo base responde a uma pergunta que está no nosso dataset.  
Isso servirá como **linha de base** para compararmos com o modelo ajustado.

In [ ]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"
    
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=80,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # ativa amostragem para usar temperatura
        temperature=0.7
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte após "Output:"
    resposta = full_output.split("Output:")[-1].strip()
    return resposta

# Exemplo de instrução (presente no dataset.jsonl)
test_instruction = "Qual é a principal característica técnica abordada no manual para sistemas de comandos de segurança?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

> **Observação:** O modelo base provavelmente gerará um texto genérico ou sem relação direta com a instrução, pois ainda não foi adaptado.

## 5. Tokenização do Dataset

Transformamos os textos em sequências de tokens que o modelo pode processar.  
Usaremos `padding="max_length"` e `truncation=True` para garantir que todas as amostras tenham o mesmo comprimento (192 tokens).

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=192  # alguns exemplos do dataset.jsonl têm cerca de 190 tokens
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

## 6. Preparar o Modelo para LoRA

A função `prepare_model_for_kbit_training` ativa técnicas como *gradient checkpointing* e ajusta a arquitetura para treinamento eficiente.  
É essencial quando se utiliza quantização (QLoRA), mas também é recomendada mesmo sem quantização para melhor gerenciamento de memória.

In [7]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

## 7. Configurar e Injetar LoRA

Agora definimos a configuração do LoRA:
- **r**: posto das matrizes de adaptação (quanto maior, mais capacidade, porém mais parâmetros).
- **lora_alpha**: fator de escala $\alpha$ (a atualização será multiplicada por $\alpha / r$).
- **target_modules**: os módulos do transformer onde aplicaremos LoRA. Na arquitetura LLaMA (TinyLlama), `q_proj`, `k_proj`, `v_proj` e `o_proj` são as projeções de atenção (Query, Key, Value e Output).
- **lora_dropout**: dropout aplicado às matrizes LoRA para regularização.
- **bias**: "none" significa que não treinamos os vieses.
- **task_type**: como é um modelo de linguagem causal, usamos `CAUSAL_LM`.

Em seguida, criamos o modelo PEFT com `get_peft_model`, que insere os adaptadores LoRA e **congela** o resto do modelo.

In [ ]:
lora_config = LoraConfig(
    r=16,                    # rank da decomposição
    lora_alpha=32,           # escala alpha
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # projeções de atenção do LLaMA
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False     # False = modo treinamento
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

✅ **Interpretação:** Apenas uma fração mínima do total de parâmetros será atualizada.  
No exemplo, menos de 1% dos pesos são treináveis – é a essência do PEFT.

## 8. Data Collator para Modelagem Causal

O `DataCollatorForLanguageModeling` prepara os lotes para o treinamento de linguagem causal (sem *masked language modeling*).  
Ele automaticamente desloca os rótulos para que a tarefa seja prever o próximo token.

In [9]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## 9. Argumentos de Treinamento

Definimos os hiperparâmetros do treinamento.  
Como nosso dataset é grande e com pares maiores, usaremos 5 épocas e uma taxa de aprendizado mais estável (`2e-4`).  
O `eval_steps` controla a frequência da avaliação no conjunto de validação.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",          # diretório de saída
    eval_strategy="steps",
    eval_steps=50,
    learning_rate=2e-4,               # LR menor: mais estável para um modelo maior (1,1B)
    per_device_train_batch_size=2,    # reduzido para caber em ~10GB de RAM
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,    # batch efetivo = 4, mesmo nível do original
    num_train_epochs=5,               # Possível porque o TinyLlama (1,1B) aprende muito mais rápido
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    report_to="none",               # desabilita logging para wandb/mlflow
)

## 10. Inicializar o Trainer

O `Trainer` do Hugging Face orquestra todo o ciclo de treinamento, avaliação e salvamento.

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

## 11. Treinar o Modelo

Iniciamos o treinamento.

In [ ]:
trainer.train()

## 12. Salvar o Modelo Ajustado e o Tokenizador

Ao final do treinamento, salvamos os pesos LoRA (apenas os adaptadores) e o tokenizador.

In [ ]:
model.save_pretrained("lora_finetuned_model")
tokenizer.save_pretrained("tinyllama_tokenizer")

## 13. Inferência APÓS o Fine-Tuning

Agora carregamos o modelo ajustado e comparamos sua resposta com a versão base, usando **exatamente a mesma instrução**.

In [ ]:
# Recarrega o modelo base (TinyLlama) e o tokenizador salvos
finetuned_base_model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True)
finetuned_tokenizer = AutoTokenizer.from_pretrained("tinyllama_tokenizer")

# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

# Aplica os adaptadores LoRA treinados sobre o modelo base
finetuned_model = PeftModel.from_pretrained(finetuned_base_model, "lora_finetuned_model")

In [ ]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")

## 14. Comparação e Conclusão

- **Antes do fine-tuning:** o modelo base não conhecia nosso domínio; sua resposta era genérica ou incoerente.
- **Depois do fine-tuning:** com apenas uma fração dos parâmetros treinados (via LoRA), o modelo aprendeu a estrutura desejada e gera respostas alinhadas com os exemplos fornecidos.